In [ ]:
"""
Building RAG Pipelines
Complete retrieval-augmented generation implementation
"""

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from dotenv import load_dotenv
from openai import OpenAI
import tempfile

In [23]:
import os
load_dotenv()

api_key = os.getenv('GROQ_API_KEY')

In [26]:

llm = init_chat_model(
    'openai/gpt-oss-20b',
    model_provider="openai",
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
    temperature=0.2,
)

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')   


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20315.70it/s]


In [4]:
KNOWLEDGE_BASE = """# LangChain Framework

LangChain is a framework for developing applications powered by language models. It was created by Harrison Chase in October 2022.

## Core Components

1. **Models**: LangChain supports various LLM providers including OpenAI, Anthropic, and local models.

2. **Prompts**: Templates for structuring inputs to language models.

3. **Chains**: Sequences of calls to models and other components.

4. **Agents**: Systems that use LLMs to determine which actions to take.

5. **Memory**: Components for persisting state between chain/agent calls.

## LangGraph

LangGraph is a library for building stateful, multi-actor applications. Key features:
- State management
- Cycles and loops
- Human-in-the-loop
- Persistence

## Pricing

LangChain itself is open source and free. LangSmith (the observability platform) has a free tier and paid plans starting at $39/month.

## Getting Started

Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code.
"""

In [16]:
def create_kb():
    db = 'chromadb'
    splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=70)
    doc = Document(page_content=KNOWLEDGE_BASE,metadata={"source": "langchain_knowledge_base.md"})
    chunks = splitter.split_documents(documents=[doc])

    vector_store = Chroma.from_documents(
        documents=chunks,embedding=embedding,persist_directory=db
    )
    return vector_store

In [40]:
def demo_basic_rag():

    vector_store = create_kb()
    retriever = vector_store.as_retriever(
        search_type="similarity", search_kwargs={"k": 2}
    )

    # RAG Prompt Template
    prompt = ChatPromptTemplate.from_template(
        """
Answer the question based only on the following context:

{context}

Question: {question}

Answer:


Make sure to answer in a concise manner, 
and if you don't know the answer, just say "I don't know."""
    )

    # Format retrieved docs
    def format_docs(docs):
        res = "\n\n".join([doc.page_content for doc in docs])
        print(f'format doc {res}')
        return res

    # Rag chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Test the RAG chain
    # Test
    questions = [
        # "What is LangChain?",
        # "Who created LangChain?",
        # "What is LangGraph used for?",
        "How do I deploy LangChain to AWS?"
    ]

    print("Basic RAG Demo:\n")
    for q in questions:
        answer = rag_chain.invoke(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")


In [41]:
demo_basic_rag()

Basic RAG Demo:

format doc ## Getting Started

Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code.

## Getting Started

Install with: pip install langchain langchain-openai
Create your first chain in under 10 lines of code.
Q: How do I deploy LangChain to AWS?
A: I don't know.



In [39]:
def structured_rag():
    vector_store = create_kb()
    retriever = vector_store.as_retriever(
        search_kwargs={'k':1}
    )

    prompt = ChatPromptTemplate.from_template(
            """
    Answer the question based only on the following context:
    
    {context}
    
    Question: {question}
    
    Answer:
    
    
    Make sure to answer in a concise manner, 
    Provide a structured response."""
        )

    class RAG_RESPONSE(BaseModel):
        answer :str = Field(description='answer to the question')
        confidence :str = Field(description='high , medium or low')
        sources_used : list[str] = Field(description='List of sources referenced')
        follow_up :str = Field(description='Suggested follow-up question')


    structured_llm = llm.with_structured_output(RAG_RESPONSE)
    
        # Format retrieved docs
    def format_docs(docs):
        res = "\n\n".join([f'{doc.metadata.get('source','unknown')} {doc.page_content} 'for doc in docs])
        print(f'Format docs {res}\n\n')
        return res
    
    rag_chain = (
          {'context':retriever | format_docs , 'question' : RunnablePassthrough()}
          | prompt
          | structured_llm
    )

    print("Structured RAG Demo:\n")
    result = rag_chain.invoke("What is LangGraph?")

    print(f"Answer: {result.answer}")
    print(f"Confidence: {result.confidence}")
    print(f"Sources: {result.sources_used}")
    print(f"Follow-up: {result.follow_up}")

structured_rag()
    

Structured RAG Demo:

Format docs langchain_knowledge_base.md 5. **Memory**: Components for persisting state between chain/agent calls.

## LangGraph

LangGraph is a library for building stateful, multi-actor applications. Key features:
- State management
- Cycles and loops
- Human-in-the-loop
- Persistence

## Pricing

LangChain itself is open source and free. LangSmith (the observability platform) has a free tier and paid plans starting at $39/month.

## Getting Started 


Answer: LangGraph is a library for building stateful, multi‑actor applications that supports state management, cycles and loops, human‑in‑the‑loop interactions, and persistence.
Confidence: high
Sources: ['langchain_knowledge_base.md', 'LangGraph section']
Follow-up: What are some common use cases for LangGraph?


In [53]:
vector_store = create_kb()
retriever = vector_store.as_retriever(
        search_kwargs={'k':1}
    )
res = retriever.invoke('What is LangGraph?')
res.page_content

AttributeError: 'list' object has no attribute 'page_content'

In [38]:
res

[Document(id='3fe96f63-5894-4d05-9458-677147cf67a9', metadata={'source': 'langchain_knowledge_base.md'}, page_content='5. **Memory**: Components for persisting state between chain/agent calls.\n\n## LangGraph\n\nLangGraph is a library for building stateful, multi-actor applications. Key features:\n- State management\n- Cycles and loops\n- Human-in-the-loop\n- Persistence\n\n## Pricing\n\nLangChain itself is open source and free. LangSmith (the observability platform) has a free tier and paid plans starting at $39/month.\n\n## Getting Started')]

In [51]:
def exercise_doc():
    class DocumentQ:
        def __init__(self, document : str, source:str = 'document'):
            splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
            doc = Document(page_content= document, metadata={'source':source})
            chunks = splitter.split_documents([doc])

            self.vector_store = Chroma.from_documents(
                documents=chunks,
                embedding=embedding,)
            
            self.retriever = self.vector_store.as_retriever(
                search_kwargs={'k':1}
            )
            self.prompt = ChatPromptTemplate.from_template(
                """
Answer based on the context. .

Context: {context}
Question: {question}
""" )
            def format_doc(doc):
                return '\n\n'.join(d.page_content for d in doc)

            self.chain = (
                {'context':self.retriever | format_doc, 'question' : RunnablePassthrough()}
                | self.prompt
                | llm
                | StrOutputParser()
            )
        def ask(self,q:str):
            return self.chain.invoke(q)

    # Test
    test_doc = """
    The Python programming language was created by Guido van Rossum.
    First released in 1991, Python emphasizes code readability.
    Python 3.12 was released in October 2023 with improved error messages.
    The language is named after Monty Python, not the snake.
    """

    bot = DocumentQ(document=test_doc,source='Python_facts')

    print("Document Q&A System:\n")
    questions = [
        "Who created Python?",
        "When was Python 3.12 released?",
        "Why is Python named Python?",
    ]

    for question in questions:
        ans = bot.ask(question)
        print(f"Q: {question}")
        print(f"A: {ans}\n")

In [52]:
exercise_doc()

Document Q&A System:

Q: Who created Python?
A: Guido van Rossum.

Q: When was Python 3.12 released?
A: Python 3.12 was released in **October 2023**.

Q: Why is Python named Python?
A: Python was named after the British comedy troupe **Monty Python**.  
Guido van Rossum, the language’s creator, was a fan of the TV show *Monty Python’s Flying Circus* and chose the name to reflect that humor and light‑heartedness, rather than any reference to the snake.

